# Sample inspection - profiles one by one (per dataset)

Drill into individual samples on the **scaled + aligned** table (higher = more binder-like). Built for
the case *"some binders look like non-binders on a few metrics"*: it (1) ranks binders by how
non-binder-like they are, then (2) for any chosen sample shows exactly **which metrics** put it on the
non-binder side, versus the Binder / Non-Binder class means.

Works on one dataset at a time. If you load a pooled set, set `DATASET_FILTER` to inspect a single member.

## Setup

In [ ]:
import sys, os
from pathlib import Path
# find project_current (folder with config/ + analysis/) no matter where this notebook is launched
_p = Path(os.environ.get('PROJECT_ROOT', Path.cwd()))
PROJECT_ROOT = next((c for c in [_p, *_p.parents] if (c/'config').is_dir() and (c/'analysis').is_dir()), _p)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config.paths import RAW_DATA_DIR, METRIC_YAML
from config.datasets import cfg
from config.analysis import WINDOW_FAMILIES
from analysis.distributions import get_numeric_metrics, add_binder_type
from analysis.evaluation.metrics import load_metric_families, get_family
from analysis.profiles import plot_sample_profiles      # family-level overview
families = load_metric_families(METRIC_YAML)

## Load one dataset
Defaults to the current `DATASET`. Set `DATASET_FILTER = 'peptide'` (etc.) to slice one member out of a
pooled set. Uses `merged_scaled_aligned.csv` (run `run_scale.py` first).

In [ ]:
DATASET_FILTER = None    # e.g. 'peptide' to keep one dataset from a pooled set; None keeps all

base = RAW_DATA_DIR / cfg.name
sa = base / 'merged_scaled_aligned.csv'
if not sa.exists():
    raise FileNotFoundError(f'{sa} not found - run run_scale.py first.')
scaled = add_binder_type(pd.read_csv(sa))
if DATASET_FILTER is not None and 'dataset' in scaled.columns:
    scaled = scaled[scaled['dataset'] == DATASET_FILTER].reset_index(drop=True)

# metric features, minus window families (net_charge/surface_hydrophobicity: no monotonic 'binder side')
metrics = [m for m in get_numeric_metrics(scaled) if get_family(m, families) not in WINDOW_FAMILIES]

if scaled['sample'].duplicated().any():
    print('WARNING: duplicate sample ids present - set DATASET_FILTER to a single dataset for clean per-sample lookup.')
counts = scaled['binder_type'].value_counts().to_dict()
print('dataset:', cfg.name, '| filter:', DATASET_FILTER, '| rows:', len(scaled))
print('classes:', counts, '| metrics:', len(metrics))
assert counts.get('Binder', 0) and counts.get('Non-Binder', 0), 'need both Binder and Non-Binder rows to compare'

## Helpers

In [ ]:
def class_means(s, ms):
    """Per-metric Binder / Non-Binder means and separation (binder_mean - nonbinder_mean)."""
    bt = s['binder_type']
    b_mean = s.loc[bt == 'Binder', ms].mean()
    n_mean = s.loc[bt == 'Non-Binder', ms].mean()
    return b_mean, n_mean, (b_mean - n_mean)

def separating_metrics(s, ms, min_sep=0.25):
    """Metrics where binders sit at least `min_sep` SD above non-binders (aligned table)."""
    _, _, sep = class_means(s, ms)
    return [m for m in ms if sep[m] > min_sep]

def suspect_binders(s, ms, min_sep=0.25):
    """Rank Binder samples by how many separating metrics they fall on the NON-binder side of
    (below the Binder/Non-Binder midpoint). High frac = a binder that looks like a non-binder."""
    b_mean, n_mean, _ = class_means(s, ms)
    sepm = separating_metrics(s, ms, min_sep)
    thr = (b_mean[sepm] + n_mean[sepm]) / 2
    S = s[s['binder_type'] == 'Binder'].set_index('sample')[sepm]
    nbl = S < thr                                   # per (sample, metric); NaN -> False
    out = pd.DataFrame({
        'sample': S.index,
        'n_nonbinderlike': nbl.sum(axis=1).values,
        'n_scored': S.notna().sum(axis=1).values,
        'mean_z': S.mean(axis=1).round(3).values,   # low = generally non-binder-like
    })
    out['frac_nonbinderlike'] = (out['n_nonbinderlike'] / out['n_scored']).round(3)
    return out.sort_values(['frac_nonbinderlike', 'n_nonbinderlike'], ascending=False).reset_index(drop=True), sepm

def sample_metric_table(s, sample, ms):
    """Per-metric position of one sample vs the class means; most non-binder-like first."""
    b_mean, n_mean, sep = class_means(s, ms)
    row = s.set_index('sample').loc[sample, ms]
    if isinstance(row, pd.DataFrame):      # duplicate sample id -> take first
        row = row.iloc[0]
    t = pd.DataFrame({'metric': ms, 'sample_z': row.values,
                      'binder_mean': b_mean.values, 'nonbinder_mean': n_mean.values,
                      'separation': sep.values})
    mid = (t['binder_mean'] + t['nonbinder_mean']) / 2
    t['side'] = np.where(t['sample_z'] < mid, 'non-binder-like', 'binder-like')
    t.loc[t['sample_z'].isna(), 'side'] = '(missing)'
    return t.sort_values('sample_z').round(3)

def plot_sample_diagnostic(s, sample, ms, max_metrics=25):
    """Lollipop: each metric's Non-Binder mean -> Binder mean span, with the sample's value marked
    (red = fell on the non-binder side). Sorted most non-binder-like at top."""
    t = sample_metric_table(s, sample, ms)
    t = t[t['side'] != '(missing)'].head(max_metrics)
    y = np.arange(len(t))[::-1]
    fig, ax = plt.subplots(figsize=(9, max(3, 0.34 * len(t))))
    ax.hlines(y, t['nonbinder_mean'], t['binder_mean'], color='lightgrey', lw=3, zorder=1)
    ax.scatter(t['nonbinder_mean'], y, marker='|', s=220, color='#eb405d', zorder=2, label='Non-Binder mean')
    ax.scatter(t['binder_mean'], y, marker='|', s=220, color='#2ecc71', zorder=2, label='Binder mean')
    col = np.where(t['side'] == 'non-binder-like', '#eb405d', '#2ecc71')
    ax.scatter(t['sample_z'], y, s=80, color=col, edgecolor='k', lw=0.6, zorder=3)
    ax.set_yticks(y); ax.set_yticklabels(t['metric'], fontsize=8)
    ax.axvline(0, color='grey', lw=0.5, ls=':')
    ax.set_xlabel('standardized, aligned  (higher = more binder-like)')
    bt = s.set_index('sample').loc[sample, 'binder_type']
    bt = bt.iloc[0] if hasattr(bt, 'iloc') else bt
    ax.set_title(f'{sample}  [{bt}]  -  per-metric position vs class means')
    ax.legend(fontsize=8, loc='lower right')
    fig.tight_layout(); return fig

## 1. Which binders look like non-binders?

In [ ]:
suspects, sepm = suspect_binders(scaled, metrics, min_sep=0.25)
print(f'{len(sepm)} separating metrics used (binders > non-binders by >0.25 SD)')
suspects.head(15)

## 2. Inspect one sample
Set `SAMPLE` to any id (defaults to the top suspect). Left/table = which metrics look non-binder-like; the family overview shows it against the Binder / Non-Binder bands.

In [ ]:
SAMPLE = suspects.iloc[0]['sample']    # or set manually, e.g. SAMPLE = 'vhh123_target'
print('inspecting:', SAMPLE)

tbl = sample_metric_table(scaled, SAMPLE, sepm)
display(tbl[tbl['side'] == 'non-binder-like'])          # the metrics that make it look like a non-binder
plot_sample_diagnostic(scaled, SAMPLE, sepm)
plt.show()
plot_sample_profiles(scaled, samples=[SAMPLE])           # family-level profile vs group IQR bands
plt.show()

## 3. Go through the suspects one by one
Renders the diagnostic for the top-`k`; adjust `k` or replace the list with your own sample ids.

In [ ]:
k = 5
for smp in suspects['sample'].head(k):
    t = sample_metric_table(scaled, smp, sepm)
    nbl = t[t['side'] == 'non-binder-like']['metric'].tolist()
    print(f"\n=== {smp}  |  non-binder-like on {len(nbl)}/{len(sepm)} metrics: {', '.join(nbl)}")
    plot_sample_diagnostic(scaled, smp, sepm)
    plt.show()

### Notes
- 'non-binder-like on metric m' = the sample's aligned/standardized value is below the midpoint between the
  Binder and Non-Binder means on a metric that actually separates (binders > non-binders by >0.25 SD).
  Tune `min_sep`, or change the midpoint to the Non-Binder mean for a stricter definition.
- Window families (net_charge, surface_hydrophobicity) are excluded - they have no monotonic 'binder side'.
- Reads the scaled+aligned table, so 'higher = more binder-like' holds per metric; raw values are in merged.csv
  if you want to see native units for a flagged metric.